# 04 — Prompt Services

This notebook is where I separate two things that are easy to collapse: what a translation says about a **service** and what it says about a **prompt**. For this stage, I focus only on the target term `Digital Humanities`. The LLM outputs form a service × prompt grid: eight services run across four prompt variants.

I use the grid in four ways:

| | **Within prompt** | **Across prompts** |
|---|---|---|
| **Within service** | 4.1 describes each service/prompt cell as an output profile | 4.3 asks whether a service agrees with itself across prompt variants |
| **Across services** | 4.2 asks whether services converge when asked the same way | 4.4 asks whether any translation survives across the full service × prompt grid |

Notebook 02 handled inventory, failures, and review signals. Notebook 03 isolated the prompt-invariant comparison sources: Wikipedia, Google Translate, EasyNMT, and Lingvanex. Those sources stay in the data as reference anchors, but they are not part of the prompt experiment because they do not change when the prompt changes. Here I begin after that accounting layer. I treat the LLM grid itself as the object of analysis and ask how much of the output is stable across services, prompts, and families.

**Exclusion tier: Tier 1 — Service Exploration** (see [docs/exclusion_strategy.md](../docs/exclusion_strategy.md))

At this tier, I keep most errors visible. Mixed scripts, source-term echoes, placeholders, and low-confidence outputs are not automatically removed, because they are part of what I am trying to study. The only automatic suppression is `enforce_translation_rationale_pairing`, which is applied when `load_variant_df` loads the files so that a translation without a real rationale does not masquerade as a complete LLM response. I also apply manual `exclude_translation=True` entries from `manual_exclusions.csv`; those are structural failures rather than ordinary quality concerns. Automated review signals from `automated_review_signals.csv` are merged as annotations, not used as filters.

Sections:
1. Service × prompt availability and output shape
2. Within-prompt cross-service agreement
3. Across-prompt within-service stability
4. Cross-service × cross-prompt consensus
5. Source-term retention and borrowing signals
6. Judge prompt as intervention

The main scripts behind this notebook are `explore_confidence_within_variant.py`, which writes `confidence_scores.csv`, and `explore_confidence_across_variants.py`, which writes `across_variant_detail.csv`. In both cases, "confidence" means frequency-based convergence among outputs, not independent proof that a translation is correct.


In [1]:
import ast
import os
import sys
from collections import Counter

import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path

alt.data_transformers.enable('vegafusion')

PROJECT_ROOT = Path('..')
if not (PROJECT_ROOT / 'scripts' / 'utils.py').exists():
    PROJECT_ROOT = Path('.')
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.utils import read_csv_file, get_language_family, save_chart
from scripts.exploration.explore_confidence_within_variant import (
    run_confidence_evaluation, ALL_VARIANTS,
)
from scripts.exploration.explore_confidence_across_variants import (
    run_across_variant_evaluation, LLM_SERVICES, BASELINE_SERVICES,
)

DATA_DIR = PROJECT_ROOT / "datasets"
TERM      = 'Digital Humanities'
TERM_SLUG = TERM.lower().replace(' ', '_')
EVAL_DIR  = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
VARIANTS  = ALL_VARIANTS
FIGURE_DIR = PROJECT_ROOT / 'figures' / 'dhq_article'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VARIANT_LABELS = {
    'minimal':          'Minimal',
    'fluent_speaker': 'Fluent Speaker',
    'github_searcher': 'GitHub Searcher',
    'judge':            'Judge',
}
SERVICE_ORDER = ['Wikipedia', 'Google Translate', 'EasyNMT', 'Lingvanex',
                 'OpenAI', 'Claude', 'Gemini', 'DeepSeek',
                 'Llama', 'Gemma', 'Qwen', 'Mistral']
SERVICE_COLOURS = {
    'Wikipedia': '#aec7e8', 'Google Translate': '#c5b0d5',
    'EasyNMT': '#c49c94',   'Lingvanex': '#dbdb8d',
    'OpenAI': '#1f77b4',    'Claude': '#ff7f0e',
    'Gemini': '#2ca02c',    'DeepSeek': '#9467bd',
    'Llama': '#8c564b',     'Gemma': '#e377c2',
    'Qwen': '#7f7f7f',      'Mistral': '#bcbd22',
}
LLM_TRANS_COLS = {
    'Claude':   'claude_translated_term',
    'OpenAI':   'openai_translated_term',
    'Gemini':   'gemini_translated_term',
    'DeepSeek': 'deepseek_translated_term',
    'Llama':    'llama_translated_term',
    'Gemma':    'gemma_translated_term',
    'Qwen':     'qwen_translated_term',
    'Mistral':  'mistral_translated_term',
}

TIER_ORDER   = ['all_identical', 'trivial_only', 'has_content']
TIER_LABELS  = {
    'all_identical': 'All identical',
    'trivial_only':  'Trivial only (cap / whitespace)',
    'has_content':   'Content differences',
}
TIER_COLOURS = {'all_identical': '#2ca02c', 'trivial_only': '#ff7f0e', 'has_content': '#d62728'}

def classify_diff_tier(d):
    if pd.isna(d) or str(d) in ('no_differences', 'all_identical', 'unknown', ''):
        return 'all_identical'
    types = set(str(d).split(','))
    return 'trivial_only' if types <= {'capitalization', 'whitespace', 'both'} else 'has_content'


print(f'Variants: {VARIANTS}')

Variants: ['minimal', 'fluent_speaker', 'github_searcher', 'judge']


In [2]:
# ── Within-variant confidence scores ─────────────────────────────────────────
scored_df, summary_df = run_confidence_evaluation(
    data_directory_path=DATA_DIR,
    target_terms=[TERM],
    variants=VARIANTS,
    output_dir=EVAL_DIR,
)
scored_df['language_family'] = scored_df['language_code'].apply(get_language_family)
print(f'scored_df: {len(scored_df)} rows')

# ── Across-variant detail ─────────────────────────────────────────────────────
detail_path  = os.path.join(EVAL_DIR, 'across_variant_detail.csv')
summary_path = os.path.join(EVAL_DIR, 'across_variant_service_summary.csv')
if not os.path.exists(detail_path):
    print('Running explore_confidence_across_variants.py...')
    detail_df, across_summary_df = run_across_variant_evaluation(DATA_DIR, [TERM])
else:
    detail_df       = read_csv_file(detail_path)
    across_summary_df = read_csv_file(summary_path)

has_data = detail_df[detail_df['n_variants_present'] > 0].copy()
llm_df   = has_data[~has_data['is_baseline']].copy()
print(f'detail_df: {len(detail_df)} rows | llm_df: {len(llm_df)} rows')
summary_df

📊 Scoring: Digital Humanities

  ✓ Loaded 881 rows for variant 'minimal'

  ✓ Loaded 881 rows for variant 'fluent_speaker'

  ✓ Loaded 881 rows for variant 'github_searcher'

  ✓ Loaded 881 rows for variant 'judge'

...

  confidence_scores.csv  : 3524 rows

  confidence_summary.csv : 4 rows

                              Confidence Scoring Summary (LLM agreement per variant)                               
┏━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ term       ┃ variant    ┃ total_lan… ┃ language… ┃ mean_llm_… ┃ median_l… ┃ std_llm_c… ┃ cv_llm_c… ┃ llm_uniqu… ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Digital    │ minimal    │ 881        │ 881       │ 0.2299     │ 0.1429    │ 0.1536     │ 0.6681    │ 6.85       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ fluent_sp… │ 881        │ 881       │ 0.21       │ 0.1429    │ 0.1463     │ 0.6965    │ 7.01       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ github_se… │ 881        │ 881       │ 0.266      │ 0.25      │ 0.1671     │ 0.6281    │ 6.67       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ judge      │ 881        │ 881       │ 0.4936     │ 0.5       │ 0.2175     │ 0.4407    │ 4.31       │
│ Humanities │            │            │           │            │           │            │           │            │
└────────────┴────────────┴────────────┴───────────┴────────────┴───────────┴────────────┴───────────┴────────────┘

scored_df: 3524 rows
detail_df: 10572 rows | llm_df: 7048 rows


,term,variant,total_languages,languages_with_llm,mean_llm_confidence,median_llm_confidence,std_llm_confidence,cv_llm_confidence,llm_unique_candidates_mean,languages_with_baseline,mean_baseline_confidence
0,Digital Humanities,minimal,881,881,0.2299,0.1429,0.1536,0.6681,6.85,257,0.7662
1,Digital Humanities,fluent_speaker,881,881,0.2100,0.1429,0.1463,0.6965,7.01,257,0.7662
2,Digital Humanities,github_searcher,881,881,0.2660,0.2500,0.1671,0.6281,6.67,257,0.7662
3,Digital Humanities,judge,881,881,0.4936,0.5000,0.2175,0.4407,4.31,257,0.7662


In [3]:
# ── Tier 1 exclusion policy ──────────────────────────────────────────────────
# Merge automated review signals as annotations (never filter on them at Tier 1).
# Apply only exclude_translation=True from manual_exclusions — structural drops only.

from scripts.exploration.translation_classifier import (
    curate_translation,
    has_extreme_term_length,
    has_source_leakage,
    has_unicode_escape,
    is_repetition_loop,
)

flags_path = os.path.join(EVAL_DIR, 'automated_review_signals.csv')
review_signals = read_csv_file(flags_path)

FLAG_ANNOTATION_COLS = [
    'language_code',
    'has_mixed_script', 'has_placeholder_term', 'has_romanization',
    'has_source_term', 'has_script_disagreement', 'has_repetition_loop',
    'has_extreme_term_length', 'has_unicode_escape', 'has_any_mixing',
    'quality_flags', 'flag_count',
]
scored_df = scored_df.merge(review_signals[FLAG_ANNOTATION_COLS], on='language_code', how='left')

excl_path = os.path.join(EVAL_DIR, 'manual_exclusions.csv')
if os.path.exists(excl_path):
    excl_df = read_csv_file(excl_path)
    whole_drops = excl_df[excl_df['exclude_translation'] == True][['language_code', 'service']]
    tier1_drop_pairs = set(zip(whole_drops['language_code'], whole_drops['service']))

    # Languages where ALL eight LLM services are dropped → no usable LLM data
    LLM_NAMES = set(LLM_TRANS_COLS.keys())
    drops_by_lang = whole_drops.groupby('language_code')['service'].apply(set).to_dict()
    fully_excluded = {lc for lc, svcs in drops_by_lang.items() if LLM_NAMES <= svcs}
    partially_excluded = {lc for lc in drops_by_lang if lc not in fully_excluded}

    # Remove fully-excluded languages; flag partial exclusions as annotation
    scored_df = scored_df[~scored_df['language_code'].isin(fully_excluded)].copy()
    scored_df['has_tier1_drop'] = scored_df['language_code'].isin(partially_excluded)

    # Re-derive detail_df and llm_df after exclusions
    detail_df = detail_df[~detail_df['language_code'].isin(fully_excluded)].copy()
    has_data = detail_df[detail_df['n_variants_present'] > 0].copy()
    llm_df   = has_data[~has_data['is_baseline']].copy()

    print(f'Tier 1 exclusions:')
    print(f'  Whole-translation drops  : {len(whole_drops)} (language × service) pairs')
    print(f'  Fully excluded languages : {len(fully_excluded)} (all LLM services dropped)')
    print(f'  Partially excluded       : {len(partially_excluded)} (flagged; some services remain)')
    print(f'  Annotated automated review signals  : {int(scored_df["flag_count"].gt(0).sum())} languages with ≥1 flag (retained as data)')
else:
    scored_df['has_tier1_drop'] = False
    print('No manual_exclusions.csv found — skipping Tier 1 drops')


Tier 1 exclusions:
  Whole-translation drops  : 122 (language × service) pairs
  Fully excluded languages : 0 (all LLM services dropped)
  Partially excluded       : 96 (flagged; some services remain)
  Annotated automated review signals  : 3392 languages with ≥1 flag (retained as data)


## 4.1 — Service × Prompt Availability and Output Shape

The main analyses in this notebook use the Tier 1 exploratory dataset, where questionable outputs remain visible as evidence. Before describing service and prompt coverage, I apply the same three-stage diagnostic used in Notebook 03 to every populated LLM service × prompt × language cell:

- **Nominal coverage** counts every non-null translation after rationale pairing and manual structural exclusions.
- **Quality-filtered coverage** removes placeholder or unsalvageable mixed-script terms, repetition loops, extreme-length outputs, and literal Unicode escapes. `curate_translation()` may first strip a recoverable romanization or source-language prefix.
- **Search-ready coverage** additionally removes outputs containing the untranslated source phrase `Digital Humanities` or its initials `DH`.

This is a cell-level diagnostic, not the final number of GitHub queries. The search-term pipeline later consolidates duplicate terms, applies reviewed corrections and exclusions, and evaluates corroboration. I keep the unfiltered cells in the rest of this notebook because the failures and source echoes are themselves part of the prompt-service comparison.


In [4]:
def _llm_filter_stage(value):
    text = str(value).strip()
    _, action = curate_translation(text)
    quality_failure = (
        action in ('nulled', 'placeholder')
        or is_repetition_loop(text)
        or has_extreme_term_length(text)
        or has_unicode_escape(text)
    )
    if quality_failure:
        return 'Removed by quality filters'
    if has_source_leakage(text, TERM):
        return 'Removed for source-term leakage'
    return 'Search-ready'

llm_filter_rows = []
for variant in VARIANTS:
    vdf = scored_df[scored_df['prompt_variant'] == variant]
    for service, col in LLM_TRANS_COLS.items():
        vals = vdf[col].dropna().astype(str).str.strip()
        vals = vals[~vals.str.lower().isin(['', 'nan', 'none'])]
        outcomes = vals.map(_llm_filter_stage)
        nominal = len(vals)
        quality_filtered = int((outcomes != 'Removed by quality filters').sum())
        search_ready = int((outcomes == 'Search-ready').sum())
        llm_filter_rows.append({
            'service': service,
            'variant': variant,
            'variant_label': VARIANT_LABELS[variant],
            'nominal': nominal,
            'quality_filtered': quality_filtered,
            'search_ready': search_ready,
            'quality_removed': nominal - quality_filtered,
            'source_leakage_removed': quality_filtered - search_ready,
            'search_ready_pct': round(100 * search_ready / nominal, 1) if nominal else 0,
        })

llm_filter_df = pd.DataFrame(llm_filter_rows)
llm_filter_summary = pd.DataFrame({
    'stage': ['Nominal coverage', 'Quality-filtered', 'Search-ready'],
    'translation_cells': [
        llm_filter_df['nominal'].sum(),
        llm_filter_df['quality_filtered'].sum(),
        llm_filter_df['search_ready'].sum(),
    ],
})
llm_filter_summary['retained_pct_of_nominal'] = (
    100 * llm_filter_summary['translation_cells'] / llm_filter_summary.loc[0, 'translation_cells']
).round(1)
display(llm_filter_summary)

filter_heatmap = alt.Chart(llm_filter_df).mark_rect().encode(
    x=alt.X('variant_label:N', sort=[VARIANT_LABELS[v] for v in VARIANTS], title='Prompt variant'),
    y=alt.Y('service:N', sort=[s for s in SERVICE_ORDER if s in LLM_TRANS_COLS], title=None),
    color=alt.Color('search_ready_pct:Q', title='Search-ready (%)', scale=alt.Scale(domain=[0, 100], scheme='blues')),
).properties(width=360, height=230, title='LLM translation cells retained after search-ready filtering')
filter_labels = filter_heatmap.mark_text(fontSize=10).encode(
    text=alt.Text('search_ready_pct:Q', format='.1f'),
    color=alt.condition(alt.datum.search_ready_pct >= 70, alt.value('white'), alt.value('black')),
)
llm_search_ready_chart = filter_heatmap + filter_labels
display(llm_search_ready_chart)
save_chart(llm_search_ready_chart, FIGURE_DIR / 'llm_search_ready_retention_by_service_and_prompt.png')

llm_filter_detail = llm_filter_df.sort_values(['variant', 'service'])[[
    'service', 'variant_label', 'nominal', 'quality_filtered', 'search_ready',
    'quality_removed', 'source_leakage_removed', 'search_ready_pct',
]]
display(llm_filter_detail)


,stage,translation_cells,retained_pct_of_nominal
0,Nominal coverage,27449,100.0
1,Quality-filtered,27219,99.2
2,Search-ready,25849,94.2


alt.LayerChart(...)

,service,variant_label,nominal,quality_filtered,search_ready,quality_removed,source_leakage_removed,search_ready_pct
8,Claude,Fluent Speaker,881,881,856,0,25,97.2
11,DeepSeek,Fluent Speaker,879,877,862,2,15,98.1
10,Gemini,Fluent Speaker,865,864,852,1,12,98.5
13,Gemma,Fluent Speaker,880,863,860,17,3,97.7
12,Llama,Fluent Speaker,877,867,854,10,13,97.4
15,Mistral,Fluent Speaker,881,871,830,10,41,94.2
9,OpenAI,Fluent Speaker,631,631,594,0,37,94.1
14,Qwen,Fluent Speaker,881,857,851,24,6,96.6
16,Claude,GitHub Searcher,880,880,707,0,173,80.3
19,DeepSeek,GitHub Searcher,875,875,749,0,126,85.6


### Tier 1 Coverage and Output Shape

I start by treating each service/prompt combination as its own unit. With one run per combination, there is no within-cell agreement to measure. Instead I ask: how many languages did this cell cover after rationale pairing and Tier 1 structural exclusions, how long were the outputs, and how much did output length vary by family and prompt? This gives me the descriptive baseline before I ask whether services or prompts agree.


In [5]:
llm_only = {s: c for s, c in LLM_TRANS_COLS.items()}
fingerprint_rows = []
for variant in VARIANTS:
    vdf = scored_df[scored_df['prompt_variant'] == variant]
    for svc, col in llm_only.items():
        if col not in vdf.columns:
            continue
        vals = vdf[col].dropna()
        vals = vals[~vals.astype(str).str.strip().isin(['', 'nan'])]
        n_langs    = len(vals)
        mean_wc    = vals.astype(str).str.split().str.len().mean() if n_langs else 0
        # Confidence when this service is the only one being compared to itself
        svc_conf   = vdf.loc[vals.index, 'llm_confidence'].mean() if n_langs else float('nan')
        fingerprint_rows.append({
            'service': svc, 'variant': variant,
            'n_languages': n_langs,
            'mean_word_count': round(mean_wc, 2) if n_langs else 0,
            'mean_llm_confidence': round(svc_conf, 3) if n_langs else float('nan'),
        })

fp_df = pd.DataFrame(fingerprint_rows)

llm_order = ['Claude', 'OpenAI', 'Gemini', 'DeepSeek', 'Llama', 'Gemma', 'Qwen', 'Mistral']

cov_heat = alt.Chart(fp_df).mark_rect().encode(
    x=alt.X('variant:N', sort=VARIANTS, title=None,
             axis=alt.Axis(labelAngle=-20)),
    y=alt.Y('service:N', sort=llm_order, title=None),
    color=alt.Color('n_languages:Q', title='languages covered',
                    scale=alt.Scale(scheme='blues')),
    tooltip=['service:N', 'variant:N', 'n_languages:Q',
             alt.Tooltip('mean_word_count:Q', format='.1f'),
             alt.Tooltip('mean_llm_confidence:Q', format='.3f')],
).properties(width=260, height=160, title='Coverage (languages)')
cov_text = cov_heat.mark_text(fontSize=10).encode(
    text='n_languages:Q',
    color=alt.condition(alt.datum.n_languages > 800,
                        alt.value('white'), alt.value('black')))

wc_heat = alt.Chart(fp_df).mark_rect().encode(
    x=alt.X('variant:N', sort=VARIANTS, title=None,
             axis=alt.Axis(labelAngle=-20)),
    y=alt.Y('service:N', sort=llm_order, title=None),
    color=alt.Color('mean_word_count:Q', title='mean word count',
                    scale=alt.Scale(scheme='oranges')),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('mean_word_count:Q', format='.2f')],
).properties(width=260, height=160, title='Mean word count')
wc_text = wc_heat.mark_text(fontSize=10).encode(
    text=alt.Text('mean_word_count:Q', format='.1f'),
    color=alt.condition(alt.datum.mean_word_count > 3.5,
                        alt.value('white'), alt.value('black')))

(cov_heat + cov_text) | (wc_heat + wc_text)

alt.HConcatChart(...)

### Coverage Across Prompt Variants

This is the prompt-sensitive version of the availability view from Notebook 02. Each line follows one LLM service across prompt variants. I use it to see whether a service's usable coverage changes when the framing changes, before moving into agreement and consensus metrics.


In [6]:
coverage_line = alt.Chart(fp_df).mark_line(opacity=0.55, strokeWidth=1.8).encode(
    x=alt.X("variant:N", sort=VARIANTS, title="Prompt variant"),
    y=alt.Y("n_languages:Q", title="languages covered", scale=alt.Scale(zero=False)),
    color=alt.Color(
        "service:N",
        sort=llm_order,
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()), range=list(SERVICE_COLOURS.values())),
        title="Service",
    ),
    detail="service:N",
)
coverage_points = alt.Chart(fp_df).mark_point(filled=True, size=75).encode(
    x=alt.X("variant:N", sort=VARIANTS, title="Prompt variant"),
    y=alt.Y("n_languages:Q", title="languages covered", scale=alt.Scale(zero=False)),
    color=alt.Color(
        "service:N",
        sort=llm_order,
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()), range=list(SERVICE_COLOURS.values())),
        title="Service",
    ),
    tooltip=[
        "service:N",
        "variant:N",
        alt.Tooltip("n_languages:Q", title="languages covered"),
        alt.Tooltip("mean_word_count:Q", format=".2f", title="mean word count"),
        alt.Tooltip("mean_llm_confidence:Q", format=".3f", title="mean LLM confidence"),
    ],
)

display((coverage_line + coverage_points).properties(
    width=460,
    height=260,
    title="LLM coverage by service across prompt variants",
))


alt.LayerChart(...)

### Prompt Variant Synthesis

Before detailed agreement metrics, I summarize what each prompt variant changes at the aggregate level: average service coverage, output length, within-prompt convergence, unique candidate count, and source-term retention. I read these as descriptive signals, not as a ranking of prompts. A prompt can increase convergence for reasons I would not want to treat as success, especially if it encourages source-term borrowing.


In [7]:
variant_summary_rows = []
for variant in VARIANTS:
    vdf = scored_df[scored_df['prompt_variant'] == variant].copy()
    fp_v = fp_df[fp_df['variant'] == variant]

    source_echo_cells = 0
    source_overlap_cells = 0
    translated_cells = 0
    for _, row in vdf.iterrows():
        for svc, col in LLM_TRANS_COLS.items():
            val = row.get(col)
            if pd.isna(val):
                continue
            val = str(val).strip()
            if not val or val.lower() in ('nan', 'none'):
                continue
            translated_cells += 1
            val_lower = val.lower()
            if val_lower == TERM.lower():
                source_echo_cells += 1
            elif 'digital' in val_lower or 'humanities' in val_lower:
                source_overlap_cells += 1

    variant_summary_rows.append({
        'variant': variant,
        'mean_service_coverage': round(fp_v['n_languages'].mean(), 1),
        'min_service_coverage': int(fp_v['n_languages'].min()),
        'max_service_coverage': int(fp_v['n_languages'].max()),
        'mean_word_count': round(fp_v['mean_word_count'].mean(), 2),
        'mean_llm_confidence': round(vdf['llm_confidence'].mean(), 3),
        'mean_unique_candidates': round(vdf['llm_unique_candidates'].mean(), 2),
        'translated_cells': translated_cells,
        'exact_source_echo_rate': round(source_echo_cells / translated_cells, 3) if translated_cells else 0,
        'partial_source_overlap_rate': round(source_overlap_cells / translated_cells, 3) if translated_cells else 0,
    })

variant_synthesis_df = pd.DataFrame(variant_summary_rows)
display(variant_synthesis_df)

synth_long = variant_synthesis_df.melt(
    id_vars='variant',
    value_vars=['mean_service_coverage', 'mean_llm_confidence', 'mean_unique_candidates', 'exact_source_echo_rate', 'partial_source_overlap_rate'],
    var_name='signal',
    value_name='value',
)
SIGNAL_LABELS = {
    'mean_service_coverage': 'Mean service coverage',
    'mean_llm_confidence': 'Mean LLM confidence',
    'mean_unique_candidates': 'Mean unique candidates',
    'exact_source_echo_rate': 'Exact source-echo rate',
    'partial_source_overlap_rate': 'Partial source-overlap rate',
}
synth_long['signal_label'] = synth_long['signal'].map(SIGNAL_LABELS)
synth_long['variant_label'] = synth_long['variant'].map(VARIANT_LABELS)

variant_synthesis_chart = alt.Chart(synth_long).mark_bar().encode(
    x=alt.X('variant_label:N', sort=[VARIANT_LABELS[v] for v in VARIANTS], title='Prompt variant'),
    y=alt.Y('value:Q', title='Value'),
    color=alt.Color('variant:N', sort=VARIANTS, legend=None, scale=alt.Scale(scheme='tableau10')),
    column=alt.Column('signal_label:N', title=None, sort=list(SIGNAL_LABELS.values())),
    tooltip=['variant:N', 'signal:N', alt.Tooltip('value:Q', format='.3f')],
).properties(width=130, height=150, title='Prompt variant aggregate signals').resolve_scale(y='independent')
display(variant_synthesis_chart)
save_chart(variant_synthesis_chart, FIGURE_DIR / 'prompt_variant_aggregate_signals.png')


,variant,mean_service_coverage,min_service_coverage,max_service_coverage,mean_word_count,mean_llm_confidence,mean_unique_candidates,translated_cells,exact_source_echo_rate,partial_source_overlap_rate
0,minimal,843.5,787,881,2.49,0.230,6.85,6748,0.053,0.199
1,fluent_speaker,846.9,631,881,2.62,0.210,7.01,6775,0.016,0.204
2,github_searcher,861.6,786,881,2.32,0.266,6.67,6893,0.090,0.195
3,judge,879.1,869,881,2.68,0.494,4.31,7033,0.025,0.188


alt.Chart(...)

### Word-Count Consistency by Service, Prompt, and Family

Word count is a coarse proxy for output strategy. It matters here because every service runs under multiple prompt variants. A high coefficient of variation (CV = standard deviation divided by mean word count) suggests that a family, service, or prompt is producing unstable output lengths.

The family labels in this notebook come through `get_language_family()`. When the reviewed metadata layer exists, that helper uses `family_name_analysis`, so these charts follow the family-level labels reviewed in Notebook 01 rather than the raw pre-review family names.

I label sign languages as **"Sign languages (gloss)"** because these systems usually produce a Latin-script gloss or description rather than a signed lexical translation. That makes their word-count behavior different from spoken/written language rows.


In [8]:
wc_rows = []
for service, col in LLM_TRANS_COLS.items():
    if col not in scored_df.columns:
        continue
    mask = scored_df[col].notna() & ~scored_df[col].astype(str).str.strip().isin(["", "nan"])
    sub = scored_df.loc[mask, ["language_code", "language_name", "language_family", "prompt_variant"]].copy()
    sub["word_count"] = scored_df.loc[mask, col].astype(str).str.split().str.len().values
    sub["service"] = service
    wc_rows.append(sub)

wc_long = pd.concat(wc_rows, ignore_index=True)
wc_long["family_grouped"] = wc_long["language_family"].replace(
    {"Sign languages": "Sign languages (gloss)"}
)

var_rows = []
for (family, service, variant), grp in wc_long.groupby(["family_grouped", "service", "prompt_variant"]):
    mean_wc = grp["word_count"].mean()
    std_wc = grp["word_count"].std()
    cv = std_wc / mean_wc if mean_wc > 0 else np.nan
    var_rows.append({
        "family": family,
        "service": service,
        "variant": variant,
        "n": len(grp),
        "mean_wc": round(mean_wc, 2),
        "std_wc": round(std_wc, 2),
        "cv": round(cv, 3),
    })
var_df = pd.DataFrame(var_rows)


In [9]:
fam_cv = (
    wc_long.groupby("family_grouped")["word_count"]
    .agg(n="count", mean="mean", std="std")
    .assign(cv=lambda d: d["std"] / d["mean"])
    .dropna(subset=["cv"])
    .sort_values("cv", ascending=False)
    .reset_index()
    .round(3)
)

fam_bar = alt.Chart(fam_cv).mark_bar().encode(
    y=alt.Y("family_grouped:N", sort=alt.EncodingSortField("cv", order="descending"), title=None),
    x=alt.X("cv:Q", title="CV (std / mean word count)"),
    color=alt.Color("cv:Q", scale=alt.Scale(scheme="reds"), legend=None),
    tooltip=[
        "family_grouped:N",
        alt.Tooltip("cv:Q", format=".3f", title="CV"),
        alt.Tooltip("mean:Q", format=".2f", title="mean words"),
        alt.Tooltip("n:Q", title="translation rows"),
    ],
).properties(
    width=400,
    height=max(240, len(fam_cv) * 14),
    title="LLM word-count variability by family — collapsed across services and prompts",
)

display(fam_bar)


alt.Chart(...)

### Family × Prompt × Service CV Heatmap

This heatmap keeps the family, prompt, and service dimensions together. Rows are sorted so high-variability families appear first. Undefined CV cells are dropped when there are too few observations to estimate variation. I use this as a prompt-behavior view rather than as a raw failure chart: it shows where output shape is unstable, not necessarily where translations are wrong.


In [10]:
valid_cv = var_df.dropna(subset=["cv"]).copy()

n_dropped = len(var_df) - len(valid_cv)
print(f"Dropped {n_dropped} family/service/variant cells with undefined CV before plotting.")
print(f"Heatmap includes {valid_cv['family'].nunique()} families across {valid_cv['service'].nunique()} LLM services.")

fam_order_cv = (
    valid_cv.groupby("family")["cv"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
llm_order = [s for s in SERVICE_ORDER if s in LLM_SERVICES]

heat = alt.Chart(valid_cv).mark_rect().encode(
    x=alt.X("variant:N", sort=VARIANTS, title="Prompt variant", axis=alt.Axis(labelAngle=-20)),
    y=alt.Y("family:N", sort=fam_order_cv, title="Language family"),
    color=alt.Color("cv:Q", scale=alt.Scale(scheme="reds"), title="CV"),
    tooltip=[
        "service:N",
        "family:N",
        "variant:N",
        alt.Tooltip("cv:Q", format=".3f", title="CV"),
        alt.Tooltip("mean_wc:Q", format=".2f", title="mean words"),
        alt.Tooltip("n:Q", title="translation rows"),
    ],
).properties(width=110, height=max(260, len(fam_order_cv) * 16))

text = alt.Chart(valid_cv).mark_text(fontSize=8).encode(
    x=alt.X("variant:N", sort=VARIANTS),
    y=alt.Y("family:N", sort=fam_order_cv),
    text=alt.Text("cv:Q", format=".2f"),
    color=alt.condition(alt.datum.cv > 0.4, alt.value("white"), alt.value("black")),
)

chart = alt.layer(heat, text, data=valid_cv).facet(
    column=alt.Column("service:N", sort=llm_order, title=None),
    columns=4,
    title="Translation word-count CV by family × prompt × service",
)

display(chart)


Dropped 0 family/service/variant cells with undefined CV before plotting.
Heatmap includes 1 families across 8 LLM services.


alt.FacetChart(...)

The preceding views describe whether outputs exist and what shape they take. The next sections ask whether those outputs agree: first across services within the same prompt, then across prompts within the same service, and finally across the full service × prompt grid.


## 4.2 — Within-Prompt Cross-Service Agreement

For each prompt variant, I ask how much the eight LLM services agree when they are asked the same way. High agreement means several services converged on the same string under the same framing. Low agreement means the services produced genuinely different candidates even before prompt variation is considered.

The key metric is **LLM confidence**: the fraction of LLM services agreeing on the most common translation within a language/prompt variant. I treat this as convergence evidence, not as proof of correctness. A high value means the services landed on the same string under the same prompt; it does not tell me whether that string is well attested, locally preferred, or just a shared borrowing.


In [11]:
plot_data = scored_df[scored_df['llm_total_services'] > 0].copy()

box = alt.Chart(plot_data).mark_boxplot(extent='min-max').encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]),
            title='LLM confidence'),
    color=alt.Color('prompt_variant:N', sort=VARIANTS, legend=None),
    tooltip=['prompt_variant:N', 'llm_confidence:Q'],
).properties(width=300, height=280,
             title='LLM confidence distribution per variant')

cands = (plot_data.groupby('prompt_variant')['llm_unique_candidates']
         .mean().reset_index()
         .rename(columns={'llm_unique_candidates': 'mean_unique_candidates'}))
bar = alt.Chart(cands).mark_bar().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('mean_unique_candidates:Q', title='mean unique candidates',
            scale=alt.Scale(domain=[0, 4])),
    color=alt.Color('prompt_variant:N', sort=VARIANTS, legend=None),
    tooltip=['prompt_variant:N',
             alt.Tooltip('mean_unique_candidates:Q', format='.2f')],
).properties(width=300, height=280,
             title='Mean unique candidates (4 = all models differ)')
bar_text = bar.mark_text(dy=-8, fontSize=10).encode(
    text=alt.Text('mean_unique_candidates:Q', format='.2f'))

box | (bar + bar_text)

alt.HConcatChart(...)

In [12]:
heatmap_data = (
    scored_df[scored_df['llm_total_services'] > 0]
    .groupby(['language_family', 'prompt_variant'])['llm_confidence']
    .mean().reset_index().rename(columns={'llm_confidence': 'mean_confidence'})
)
family_order = (
    heatmap_data.groupby('language_family')['mean_confidence']
    .mean().sort_values(ascending=False).index.tolist()
)

rect = alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('language_family:N', sort=family_order, title=None),
    color=alt.Color('mean_confidence:Q',
                    scale=alt.Scale(scheme='redyellowgreen', domain=[0, 1]),
                    title='mean LLM confidence'),
    tooltip=['language_family:N', 'prompt_variant:N',
             alt.Tooltip('mean_confidence:Q', format='.2f')],
)
rect_text = rect.mark_text(fontSize=9).encode(
    text=alt.Text('mean_confidence:Q', format='.2f'),
    color=alt.condition(alt.datum.mean_confidence > 0.5,
                        alt.value('black'), alt.value('white')))

(rect + rect_text).properties(
    width=350, height=500,
    title='Mean LLM confidence — language family × prompt variant')

alt.LayerChart(...)

In [13]:
fully_divergent = (
    scored_df[
        (scored_df['llm_total_services'] >= 4) &
        (scored_df['llm_confidence'] <= 0.25)
    ]
    .groupby('language_code').size()
    .reset_index(name='variants_fully_divergent')
    .sort_values(['variants_fully_divergent', 'language_code'], ascending=[False, True])
)
print(f'Fully divergent in >=1 variant:  {len(fully_divergent)}')
print(f'Fully divergent in all 4 variants: '
      f'{(fully_divergent["variants_fully_divergent"] == 4).sum()}')

if not fully_divergent.empty:
    divergent_detail = fully_divergent.merge(
        scored_df[['language_code', 'language_name', 'language_family']].drop_duplicates(),
        on='language_code', how='left'
    )
    display(divergent_detail.head(30))


Fully divergent in >=1 variant:  768
Fully divergent in all 4 variants: 164


,language_code,variants_fully_divergent,language_name,language_family
0,ach,4,Acoli,Other
1,ady,4,Adyghe,Other
2,akk,4,Akkadian,Other
3,alt,4,Southern Altai,Other
4,ang,4,Anglo-Saxon / Old English,Other
5,atj,4,Atikamekw,Other
6,av,4,Avaric,Other
7,ay,4,Aymara,Other
8,bax,4,Bamun,Other
9,bci,4,Baoulé,Other


In [14]:
comp_mean = (
    scored_df[scored_df['llm_total_services'] > 0]
    .groupby('prompt_variant')[['llm_confidence', 'baseline_confidence']]
    .mean().reindex(VARIANTS).reset_index()
)
comp_mean = comp_mean.rename(columns={
    'llm_confidence': 'mean_llm_confidence',
    'baseline_confidence': 'mean_prompt_invariant_confidence',
})
print('Prompt-level confidence summary (LLM vs prompt-invariant-source sanity check):')
display(comp_mean)


Prompt-level confidence summary (LLM vs prompt-invariant-source sanity check):


,prompt_variant,mean_llm_confidence,mean_prompt_invariant_confidence
0,minimal,0.229917,0.223515
1,fluent_speaker,0.210047,0.223515
2,github_searcher,0.266049,0.223515
3,judge,0.493615,0.223515


In [15]:
diff_df = scored_df[scored_df['llm_total_services'] >= 2].copy()
diff_df['diff_tier']  = diff_df['llm_difference_types'].apply(classify_diff_tier)
diff_df['tier_label'] = diff_df['diff_tier'].map(TIER_LABELS)

tier_counts = (diff_df['diff_tier'].value_counts()
               .reset_index().rename(columns={'count': 'n'}))
tier_counts.columns = ['diff_tier', 'n']
tier_counts['pct'] = (tier_counts['n'] / len(diff_df) * 100).round(1)
tier_counts['label'] = tier_counts['diff_tier'].map(TIER_LABELS)
for _, r in tier_counts.sort_values('diff_tier').iterrows():
    print(f"  {r['label']}: {r['n']} ({r['pct']}%)")

tier_bar = alt.Chart(tier_counts).mark_bar().encode(
    x='n:Q',
    y=alt.Y('label:N', sort=[TIER_LABELS[t] for t in TIER_ORDER], title=None),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER,
                        range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        legend=None),
).properties(width=320, height=120, title='Difference tier breakdown — within prompt')

non_identical = diff_df[diff_df['diff_tier'] != 'all_identical'].copy()
conf_box = alt.Chart(non_identical).mark_boxplot(extent='min-max').encode(
    x=alt.X('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]),
            title='LLM confidence'),
    y=alt.Y('tier_label:N',
            sort=[TIER_LABELS['trivial_only'], TIER_LABELS['has_content']],
            title=None),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER,
                        range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        legend=None),
).properties(width=320, height=120,
             title='Confidence by tier (excl. all-identical)')

tier_bar & conf_box

  All identical: 40 (1.1%)
  Content differences: 3476 (98.6%)
  Trivial only (cap / whitespace): 8 (0.2%)


alt.VConcatChart(...)

## 4.3 — Across-Prompt Within-Service Stability

Here I hold the service constant and ask whether it gives the same translation when the prompt changes. A stable service may have a more grounded answer, or it may be consistently preserving the source term. An unstable service is more prompt-sensitive: its output changes with framing.

The key metric is **agreement_rate**: the fraction of prompt variants that produced the same best translation for a language/service pair. I use this to identify languages and services where prompt wording matters most, while keeping in mind that stability can reflect either confidence or repetition.


In [16]:
summary_plot = across_summary_df.copy()
summary_plot['is_baseline'] = summary_plot['is_baseline'].astype(bool)
summary_plot = summary_plot[~summary_plot['is_baseline']].copy()
llm_service_order = [s for s in SERVICE_ORDER if s in LLM_SERVICES]

bars = alt.Chart(summary_plot).mark_bar().encode(
    x=alt.X('mean_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean agreement rate across variants'),
    y=alt.Y('service:N', sort=llm_service_order, title=None),
    color=alt.Color('service:N',
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()),
                        range=list(SERVICE_COLOURS.values())),
        legend=None),
    tooltip=['service:N', 'mean_agreement:Q', 'std_agreement:Q',
             'n_languages:Q'],
)
text = alt.Chart(summary_plot).mark_text(align='left', dx=4, fontSize=11).encode(
    x='mean_agreement:Q', y=alt.Y('service:N', sort=llm_service_order),
    text=alt.Text('mean_agreement:Q', format='.2f'))

(bars + text).properties(width=450, height=240,
    title='Service prompt stability — mean cross-variant agreement (LLM services only)')


alt.LayerChart(...)

In [17]:
llm_service_order = [s for s in SERVICE_ORDER if s in LLM_SERVICES]

alt.Chart(llm_df).mark_boxplot(extent='min-max', size=20).encode(
    x=alt.X('agreement_rate:Q', scale=alt.Scale(domain=[0, 1]),
            title='Agreement rate (per language)'),
    y=alt.Y('service:N', sort=llm_service_order, title=None),
    color=alt.Color('service:N',
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()),
                        range=list(SERVICE_COLOURS.values())),
        legend=None),
).properties(width=450, height=200,
    title='Per-language agreement rate distribution (LLM services only)')

alt.Chart(...)

In [18]:
family_service = (
    llm_df.groupby(['language_family', 'service'])['agreement_rate']
    .mean().reset_index().rename(columns={'agreement_rate': 'mean_agreement'})
)
fam_order_s3 = (
    family_service.groupby('language_family')['mean_agreement']
    .mean().sort_values(ascending=False).index.tolist()
)

alt.Chart(family_service).mark_rect().encode(
    x=alt.X('service:N', sort=llm_service_order, title=None),
    y=alt.Y('language_family:N', sort=fam_order_s3, title='Language Family'),
    color=alt.Color('mean_agreement:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 1]),
                    title='Mean agreement'),
    tooltip=['language_family:N', 'service:N',
             alt.Tooltip('mean_agreement:Q', format='.2f')],
).properties(width=350, height=720,
    title='Mean cross-variant agreement — language family × LLM service')

alt.Chart(...)

In [19]:
pair_matches = {(a, b): [] for a in VARIANTS for b in VARIANTS if a < b}
for _, row in llm_df.iterrows():
    try:
        vt = ast.literal_eval(str(row['variant_translations']))
    except Exception:
        continue
    for (a, b) in pair_matches:
        ta, tb = vt.get(a), vt.get(b)
        if ta and tb:
            pair_matches[(a, b)].append(
                1 if str(ta).strip().lower() == str(tb).strip().lower() else 0)

pair_rows = []
for (a, b), matches in pair_matches.items():
    if matches:
        sim = round(sum(matches) / len(matches), 3)
        pair_rows += [{'va': a, 'vb': b, 'sim': sim},
                      {'va': b, 'vb': a, 'sim': sim}]
for v in VARIANTS:
    pair_rows.append({'va': v, 'vb': v, 'sim': 1.0})

pair_df = pd.DataFrame(pair_rows)
pair_df['la'] = pair_df['va'].map(VARIANT_LABELS)
pair_df['lb'] = pair_df['vb'].map(VARIANT_LABELS)
label_order = list(VARIANT_LABELS.values())

heat = alt.Chart(pair_df).mark_rect().encode(
    x=alt.X('la:N', sort=label_order, title=None),
    y=alt.Y('lb:N', sort=label_order, title=None),
    color=alt.Color('sim:Q', scale=alt.Scale(scheme='greens', domain=[0, 1]),
                    title='Fraction same'),
    tooltip=['la:N', 'lb:N', alt.Tooltip('sim:Q', format='.2f')],
)
heat_text = heat.mark_text(fontSize=11).encode(
    text=alt.Text('sim:Q', format='.2f'),
    color=alt.condition(alt.datum.sim > 0.6,
                        alt.value('white'), alt.value('black')))
(heat + heat_text).properties(width=300, height=300,
    title='Pairwise prompt-variant similarity (LLM services, normalised match)')

alt.LayerChart(...)

In [20]:
lang_stability = (
    llm_df.groupby(['language_code', 'language_name', 'language_family'])['agreement_rate']
    .mean().reset_index().rename(columns={'agreement_rate': 'mean_llm_agreement'})
    .sort_values('mean_llm_agreement')
)

print('Least stable languages (lowest mean LLM cross-variant agreement):')
display(lang_stability.head(30))


Least stable languages (lowest mean LLM cross-variant agreement):


,language_code,language_name,language_family,mean_llm_agreement
213,ewo,Ewondo,Atlantic-Congo,0.250000
109,bua,Buriat,Mongolic-Khitan,0.250000
627,rar,Rarotongan,Austronesian languages,0.250000
5,ach,Acoli,Nilotic,0.250000
687,shi,Tachelhit,Afro-Asiatic languages,0.250000
45,atj,Atikamekw,Algic,0.260412
227,fon,Fon,Atlantic-Congo,0.260412
321,ike,Eastern Canadian Inuktitut,Eskimo-Aleut languages,0.260412
16,akk,Akkadian,Afro-Asiatic languages,0.260412
548,nhw,Western Huasteca Nahuatl,Uto-Aztecan,0.260412


In [21]:
wiki_df = (detail_df[detail_df['service'] == 'Wikipedia']
           [['language_code', 'best_candidate']].copy())
wiki_df['has_wikipedia'] = (wiki_df['best_candidate'].notna() &
                            (wiki_df['best_candidate'].astype(str).str.strip() != 'nan'))

lang_wiki = lang_stability.merge(wiki_df[['language_code', 'has_wikipedia']],
                                 on='language_code', how='left')
lang_wiki['has_wikipedia']   = lang_wiki['has_wikipedia'].fillna(False)
lang_wiki['wikipedia_label'] = lang_wiki['has_wikipedia'].map({
    True:  'Wikipedia translation exists',
    False: 'No Wikipedia translation',
})
print(lang_wiki.groupby('wikipedia_label')['mean_llm_agreement']
      .agg(['mean', 'median', 'count']).round(3))

base = alt.Chart(lang_wiki)
colours = alt.Scale(
    domain=['Wikipedia translation exists', 'No Wikipedia translation'],
    range=['#2ca02c', '#d62728'],
)
scatter = base.mark_circle(opacity=0.6, size=60).encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean LLM cross-variant agreement'),
    color=alt.Color('wikipedia_label:N', scale=colours, title=None),
    tooltip=['language_name:N', 'language_family:N',
             alt.Tooltip('mean_llm_agreement:Q', format='.2f'),
             'wikipedia_label:N'],
).properties(width=420, height=220)
strip = base.mark_tick(thickness=2, bandSize=12).encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1])),
    y=alt.Y('wikipedia_label:N', title=None),
    color=alt.Color('wikipedia_label:N', scale=colours, legend=None),
).properties(width=420, height=80,
             title='LLM stability vs Wikipedia coverage')
alt.vconcat(scatter, strip).resolve_scale(color='shared')

                               mean  median  count
wikipedia_label                                   
No Wikipedia translation      0.434   0.406    841
Wikipedia translation exists  0.710   0.719     40


alt.VConcatChart(...)

## 4.4 — Cross-Service × Cross-Prompt Consensus

The strongest LLM-side signal is a translation that appears across multiple service/prompt cells at once. A term that appears in Claude/minimal, Gemini/github_searcher, and OpenAI/fluent_speaker is stronger convergence evidence than a term that appears in only one cell.

**Consensus count** is the number of service × prompt cells that agree on the most common translation for a language. The maximum is 32: eight LLM services times four prompt variants. **Consensus rate** divides that count by the number of cells with data.

I still treat consensus cautiously. Models can converge because a term is well established, but they can also converge on an English borrowing, a shared calque, or a common hallucination. The source-term section below checks for one especially important version of that problem: agreement inflated by retaining `Digital Humanities` or its English components.


In [22]:
consensus_rows = []
for lang_code, grp in scored_df.groupby('language_code'):
    cells = []
    for _, row in grp.iterrows():
        variant = row['prompt_variant']
        for svc, col in LLM_TRANS_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                cells.append({'service': svc, 'variant': variant,
                              'term': str(val).strip()})
    if not cells:
        continue

    total_cells = len(cells)
    term_counts  = Counter(c['term'] for c in cells)
    top_term, top_count = term_counts.most_common(1)[0]
    agreeing = [c for c in cells if c['term'] == top_term]

    consensus_rows.append({
        'language_code':     lang_code,
        'language_name':     grp['language_name'].iloc[0],
        'language_family':   get_language_family(lang_code),
        'top_term':          top_term,
        'consensus_count':   top_count,
        'total_cells':       total_cells,
        'consensus_rate':    round(top_count / total_cells, 3),
        'n_services_agree':  len(set(c['service'] for c in agreeing)),
        'n_variants_agree':  len(set(c['variant'] for c in agreeing)),
        'n_unique_terms':    len(term_counts),
    })

consensus_df = pd.DataFrame(consensus_rows)

print(f'Languages with LLM data: {len(consensus_df)}')
print(f'Full consensus (rate = 1.0, all cells agree): '
      f'{(consensus_df["consensus_rate"] == 1.0).sum()}')
print(f'Strong consensus (rate ≥ 0.75): '
      f'{(consensus_df["consensus_rate"] >= 0.75).sum()}')
print(f'Weak consensus (rate < 0.5):  '
      f'{(consensus_df["consensus_rate"] < 0.5).sum()}')
print()
print('Service × variant coverage per language (total_cells distribution):')
print(consensus_df['total_cells'].describe().round(1))

Languages with LLM data: 881
Full consensus (rate = 1.0, all cells agree): 1
Strong consensus (rate ≥ 0.75): 11
Weak consensus (rate < 0.5):  776

Service × variant coverage per language (total_cells distribution):
count    881.0
mean      31.2
std        1.2
min       22.0
25%       31.0
50%       32.0
75%       32.0
max       32.0
Name: total_cells, dtype: float64


In [23]:
# ── Distribution of consensus rate ──────────────────────────────────────────
hist = alt.Chart(consensus_df).mark_bar().encode(
    x=alt.X('consensus_rate:Q', bin=alt.Bin(extent=[0, 1], step=0.0625),
            title='Consensus rate (fraction of cells agreeing on top term)'),
    y=alt.Y('count():Q', title='Languages'),
    color=alt.condition(
        alt.datum.consensus_rate >= 0.75,
        alt.value('#2ca02c'), alt.value('#d62728')),
    tooltip=[alt.Tooltip('consensus_rate:Q',
                         bin=alt.Bin(extent=[0,1], step=0.0625)),
             'count():Q'],
).properties(width=420, height=240,
    title='Cross-service × cross-prompt consensus rate distribution')

# ── Breakdown: how many services and variants participate in consensus ────────
svc_bar = alt.Chart(consensus_df).mark_bar().encode(
    x=alt.X('n_services_agree:O', title='Services agreeing on top term'),
    y=alt.Y('count():Q', title='Languages'),
    color=alt.Color('n_services_agree:O',
                    scale=alt.Scale(scheme='blues'), legend=None),
    tooltip=['n_services_agree:O', 'count():Q'],
).properties(width=200, height=200, title='Services in consensus')

var_bar = alt.Chart(consensus_df).mark_bar().encode(
    x=alt.X('n_variants_agree:O', title='Variants agreeing on top term'),
    y=alt.Y('count():Q', title='Languages'),
    color=alt.Color('n_variants_agree:O',
                    scale=alt.Scale(scheme='purples'), legend=None),
    tooltip=['n_variants_agree:O', 'count():Q'],
).properties(width=200, height=200, title='Variants in consensus')

# ── Family breakdown — which families have strong consensus ──────────────────
fam_cons = (
    consensus_df.groupby('language_family')['consensus_rate']
    .mean().reset_index()
    .sort_values('consensus_rate', ascending=False)
)
fam_bar = alt.Chart(fam_cons).mark_bar().encode(
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField('consensus_rate', order='descending'),
            title=None),
    x=alt.X('consensus_rate:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean consensus rate'),
    color=alt.condition(
        alt.datum.consensus_rate >= 0.5,
        alt.value('#2ca02c'), alt.value('#d62728')),
    tooltip=['language_family:N',
             alt.Tooltip('consensus_rate:Q', format='.2f')],
).properties(width=300, height=620, title='Mean consensus rate by language family')

consensus_overview_chart = hist & (svc_bar | var_bar | fam_bar)
display(consensus_overview_chart)
consensus_distribution_chart = hist
save_chart(consensus_distribution_chart, FIGURE_DIR / 'cross_service_cross_prompt_consensus_distribution.png')

alt.VConcatChart(...)

## 4.5 — Source-Term Retention and Borrowing Signals

This section asks how often LLMs retain `Digital Humanities` or parts of the English source term instead of producing a fully translated expression. I do not assume that retention is always wrong. In some language communities, English borrowing may be the conventional form. But retention can inflate agreement and consensus, so I treat it as an interpretive check after the agreement metrics.

I use two complementary measures:

- **Exact echo rate**: the translation string equals the source term exactly, ignoring case.
- **Partial overlap rate**: the translation contains `digital` or `humanities` as a substring but is not an exact echo.

For exact echoes, I also inspect the rationale. A model that says it retained the English term because there is no established local equivalent is different from a model that simply repeats the source term with a generic rationale. This section therefore does not automatically classify source-term retention as an error; it marks where agreement needs closer interpretation.


In [24]:
from scripts.utils import apply_manual_analysis_exclusions, load_manual_exclusions
from scripts.exploration.explore_confidence_within_variant import load_variant_df

excl_eval_dir = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')
analysis_terms, search_terms, corrections = load_manual_exclusions(excl_eval_dir)

SRC_TERM  = TERM   # "Digital Humanities"
SRC_LOWER = SRC_TERM.lower()
SRC_WORDS = {"digital", "humanities"}

# Derive rationale column names from term column names
# e.g. 'claude_translated_term' → 'claude_translation_rationale'
LLM_RAT_COLS = {
    svc: col.replace('translated_term', 'translation_rationale')
    for svc, col in LLM_TRANS_COLS.items()
}

passthrough_rows = []

for variant in VARIANTS:
    vdf = load_variant_df(DATA_DIR, TERM_SLUG, variant)
    if vdf is None or vdf.empty:
        continue
    vdf = apply_manual_analysis_exclusions(vdf, analysis_terms, list(LLM_TRANS_COLS.values()))

    for svc, term_col in LLM_TRANS_COLS.items():
        rat_col = LLM_RAT_COLS.get(svc)
        if term_col not in vdf.columns:
            continue
        for _, row in vdf.iterrows():
            raw = str(row.get(term_col, '') or '').strip()
            if not raw or raw.lower() in ('nan', 'none', ''):
                continue
            exact   = raw.lower() == SRC_LOWER
            words   = set(raw.lower().split())
            partial = (not exact) and bool(words & SRC_WORDS)
            rat_val = str(row.get(rat_col, '') or '').strip() if rat_col and rat_col in vdf.columns else ''
            passthrough_rows.append({
                'language_code':   row['language_code'],
                'language_name':   row.get('language_name', row['language_code']),
                'language_family': row.get('language_family', get_language_family(row['language_code'])),
                'service': svc,
                'variant': variant,
                'translation': raw,
                'exact_echo':     exact,
                'partial_overlap': partial,
                'rationale':      rat_val,
            })

pt_df = pd.DataFrame(passthrough_rows)

total_cells = len(pt_df)
n_exact     = pt_df['exact_echo'].sum()
n_partial   = pt_df['partial_overlap'].sum()

print(f"Total (service × variant × language) cells analysed: {total_cells:,}")
print(f"  Exact echo    (= source term):              {n_exact:,}  ({n_exact/total_cells:.1%})")
print(f"  Partial overlap (DH words, not exact):      {n_partial:,}  ({n_partial/total_cells:.1%})")
print()

rate_df = (
    pt_df.groupby(['service', 'variant'])
    .agg(total=('exact_echo', 'count'),
         exact=('exact_echo', 'sum'),
         partial=('partial_overlap', 'sum'))
    .reset_index()
)
rate_df['exact_rate']   = rate_df['exact']   / rate_df['total']
rate_df['partial_rate'] = rate_df['partial'] / rate_df['total']

print("Exact echo rate by service × variant:")
pivot = rate_df.pivot(index='service', columns='variant', values='exact_rate').round(3)
print(pivot.reindex(columns=VARIANTS).to_string())

Total (service × variant × language) cells analysed: 27,343
  Exact echo    (= source term):              1,261  (4.6%)
  Partial overlap (DH words, not exact):      2,562  (9.4%)

Exact echo rate by service × variant:
variant   minimal  fluent_speaker  github_searcher  judge
service                                                  
Claude      0.105           0.028            0.197  0.108
DeepSeek    0.022           0.014            0.131  0.033
Gemini      0.080           0.014            0.069  0.015
Gemma       0.001           0.003            0.002  0.005
Llama       0.008           0.002            0.005  0.000
Mistral     0.005           0.015            0.005  0.015
OpenAI      0.210           0.057            0.291  0.020
Qwen        0.005           0.007            0.014  0.002


In [25]:
# Heatmaps: exact echo rate and partial overlap rate per service × variant
chart_df = rate_df.copy()
chart_df['exact_pct'] = (chart_df['exact_rate'] * 100).round(1)
chart_df['partial_pct'] = (chart_df['partial_rate'] * 100).round(1)
chart_df['partial_minus_exact_pct'] = (chart_df['partial_pct'] - chart_df['exact_pct']).round(1)
chart_df['variant_label'] = chart_df['variant'].map(VARIANT_LABELS)
svc_order = sorted(chart_df['service'].unique())
var_order  = [VARIANT_LABELS[v] for v in VARIANTS]

heat = alt.Chart(chart_df).mark_rect().encode(
    x=alt.X('variant_label:N', sort=var_order, title='Prompt variant'),
    y=alt.Y('service:N', sort=svc_order, title=None),
    color=alt.Color('exact_pct:Q',
                    scale=alt.Scale(scheme='orangered', domain=[0, 50]),
                    title='Exact echo %'),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('exact_pct:Q', title='Exact echo %'),
             alt.Tooltip('partial_pct:Q', title='Partial overlap %'),
             alt.Tooltip('partial_minus_exact_pct:Q', title='Partial - exact (pp)'),
             alt.Tooltip('total:Q', title='Cells')],
).properties(width=360, height=240,
    title='Source-term exact echo rate (%) per service × variant')

heat_partial = alt.Chart(chart_df).mark_rect().encode(
    x=alt.X('variant_label:N', sort=var_order, title='Prompt variant'),
    y=alt.Y('service:N', sort=svc_order, title=None),
    color=alt.Color('partial_pct:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 20]),
                    title='Partial overlap %'),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('partial_pct:Q', title='Partial overlap %'),
             alt.Tooltip('exact_pct:Q', title='Exact echo %')],
).properties(width=360, height=240,
    title='Source-term partial overlap rate (%) per service × variant')

diff_heat = alt.Chart(chart_df).mark_rect().encode(
    x=alt.X('variant_label:N', sort=var_order, title='Prompt variant'),
    y=alt.Y('service:N', sort=svc_order, title=None),
    color=alt.Color('partial_minus_exact_pct:Q',
                    scale=alt.Scale(scheme='blueorange', domainMid=0),
                    title='Partial - exact (pp)'),
    tooltip=['service:N', 'variant:N',
             alt.Tooltip('partial_minus_exact_pct:Q', title='Partial - exact (pp)'),
             alt.Tooltip('exact_pct:Q', title='Exact echo %'),
             alt.Tooltip('partial_pct:Q', title='Partial overlap %')],
).properties(width=360, height=240,
    title='Partial-overlap minus exact-echo rate (%)')

overall_corr = chart_df[['exact_rate', 'partial_rate']].corr(method='pearson').iloc[0, 1]
rank_corr = chart_df[['exact_rate', 'partial_rate']].corr(method='spearman').iloc[0, 1]
by_service_corr = (
    chart_df.groupby('service')
    .apply(lambda g: g[['exact_rate', 'partial_rate']].corr(method='pearson').iloc[0, 1] if len(g) >= 2 else float('nan'))
    .reset_index(name='pearson_corr_exact_vs_partial')
)
print(f'Across service x variant cells: Pearson r = {overall_corr:.3f}; Spearman rho = {rank_corr:.3f}')
print('By service: exact-vs-partial echo correlation across prompt variants')
display(by_service_corr)

scatter = alt.Chart(chart_df).mark_circle(size=90, opacity=0.75).encode(
    x=alt.X('exact_pct:Q', title='Exact echo %'),
    y=alt.Y('partial_pct:Q', title='Partial overlap %'),
    color=alt.Color('service:N', sort=svc_order, title='Service'),
    shape=alt.Shape('variant_label:N', sort=var_order, title='Prompt variant'),
    tooltip=['service:N', 'variant:N', 'exact_pct:Q', 'partial_pct:Q', 'partial_minus_exact_pct:Q'],
).properties(width=300, height=240, title='Exact vs partial source-term retention')

source_retention_chart = ((heat | heat_partial) & (diff_heat | scatter)).resolve_scale(color='independent')
display(source_retention_chart)
save_chart(source_retention_chart, FIGURE_DIR / 'source_term_retention_by_service_and_prompt.png')


Across service x variant cells: Pearson r = -0.205; Spearman rho = -0.043
By service: exact-vs-partial echo correlation across prompt variants


,service,pearson_corr_exact_vs_partial
0,Claude,-0.634151
1,DeepSeek,0.832808
2,Gemini,0.593877
3,Gemma,0.969095
4,Llama,0.805997
5,Mistral,0.487966
6,OpenAI,-0.700616
7,Qwen,-0.278202


alt.VConcatChart(...)

In [26]:
# ── Rationale quality for exact-echo cells ───────────────────────────────────
# Does the model explain *why* it's keeping the English term, or give a generic response?
# Simple proxy: rationale mentions "no equivalent", "untranslatable", "no direct",
# "same term", "widely used", "internationally recognised", or language name.

AWARE_PATTERNS = [
    r'no (direct |equivalent |established )?translat',
    r'no (single |established )?equivalent',
    r'untranslat',
    r'same term',
    r'widely used',
    r'international',
    r'adopted.*english',
    r'borrow',
    r'retain',
]
import re as _re
aware_re = _re.compile('|'.join(AWARE_PATTERNS), _re.IGNORECASE)

echo_df = pt_df[pt_df['exact_echo']].copy()
echo_df['rationale_aware'] = echo_df['rationale'].apply(
    lambda r: bool(aware_re.search(r)) if r else False
)
echo_df['has_rationale'] = echo_df['rationale'].apply(lambda r: bool(str(r).strip()))

n_echo   = len(echo_df)
n_aware  = echo_df['rationale_aware'].sum()
n_no_rat = (~echo_df['has_rationale']).sum()

print(f"Exact-echo cells: {n_echo}")
print(f"  Rationale explicitly justifies pass-through : {n_aware}  ({n_aware/n_echo:.0%})")
print(f"  No rationale at all                         : {n_no_rat}  ({n_no_rat/n_echo:.0%})")
print(f"  Rationale present but generic               : {n_echo-n_aware-n_no_rat}  ({(n_echo-n_aware-n_no_rat)/n_echo:.0%})")
print()

# By service
aware_by_svc = (
    echo_df.groupby('service')
    .agg(n_echo=('exact_echo','sum'), n_aware=('rationale_aware','sum'))
    .reset_index()
)
aware_by_svc['aware_rate'] = aware_by_svc['n_aware'] / aware_by_svc['n_echo']
print("Aware-rationale rate per service (for exact-echo cells):")
print(aware_by_svc.sort_values('aware_rate', ascending=False)[['service','n_echo','n_aware','aware_rate']].to_string(index=False))

Exact-echo cells: 1261
  Rationale explicitly justifies pass-through : 718  (57%)
  No rationale at all                         : 0  (0%)
  Rationale present but generic               : 543  (43%)

Aware-rationale rate per service (for exact-echo cells):
 service  n_echo  n_aware  aware_rate
  Claude     384      311    0.809896
DeepSeek     175      114    0.651429
  Gemini     150       93    0.620000
   Gemma      10        6    0.600000
  OpenAI     472      181    0.383475
    Qwen      24        5    0.208333
 Mistral      34        6    0.176471
   Llama      12        2    0.166667


## 4.6 — Judge Prompt as Intervention

The **judge** prompt is not just another wording variant. It gives each LLM the other services' minimal-variant translations before asking for its own answer. I therefore treat it as an intervention: does seeing the shared context make services converge, and if so, toward which services or clusters?

The primary comparison is **judge vs minimal** because the minimal outputs are the ones shown inside the judge setup. To avoid overstating that contrast, I also report judge-versus-other-variant summaries. This matters because the judge prompt is not independent evidence; it is a test of what happens when model outputs are allowed to condition one another.

I use two analyses:

**1. Service × service agreement matrix for judge** — how often service pairs produce the same translation in the judge variant. This shows which services cluster after shared context.

**2. Delta from minimal** — how much judge increases or decreases pairwise agreement compared with minimal. A positive delta means the judge prompt brought two services closer; a negative delta means it introduced divergence.


In [27]:
if 'analysis_terms' not in dir():
    from scripts.utils import apply_manual_analysis_exclusions, load_manual_exclusions
    analysis_terms, _, _ = load_manual_exclusions(os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation'))

from scripts.exploration.explore_confidence_within_variant import load_variant_df

variant_vdfs = {
    variant: load_variant_df(DATA_DIR, TERM_SLUG, variant)
    for variant in VARIANTS
}
_svcs = list(LLM_TRANS_COLS.keys())

def build_svc_term_map(vdf):
    vdf = apply_manual_analysis_exclusions(vdf, analysis_terms, list(LLM_TRANS_COLS.values()))
    vdf = vdf.drop_duplicates('language_code')
    result = {}
    for _, row in vdf.iterrows():
        lc = row['language_code']
        result[lc] = {}
        for svc, col in LLM_TRANS_COLS.items():
            val = str(row.get(col, '') or '').strip()
            if val and val.lower() not in ('nan', 'none'):
                result[lc][svc] = val.lower()
    return result

variant_maps = {
    variant: build_svc_term_map(vdf)
    for variant, vdf in variant_vdfs.items()
    if vdf is not None
}

def pairwise_agreement(term_map):
    counts  = {(a, b): 0 for a in _svcs for b in _svcs}
    totals  = {(a, b): 0 for a in _svcs for b in _svcs}
    for _, svc_terms in term_map.items():
        for a in _svcs:
            for b in _svcs:
                if a in svc_terms and b in svc_terms:
                    totals[(a, b)] += 1
                    if svc_terms[a] == svc_terms[b]:
                        counts[(a, b)] += 1
    result = {}
    for a in _svcs:
        result[a] = {}
        for b in _svcs:
            t = totals[(a, b)]
            result[a][b] = counts[(a, b)] / t if t > 0 else float('nan')
    return result

variant_agree = {variant: pairwise_agreement(term_map) for variant, term_map in variant_maps.items()}
judge_agree = variant_agree['judge']
minimal_agree = variant_agree['minimal']

agree_rows = []
for a in _svcs:
    for b in _svcs:
        j_val = judge_agree[a].get(b, float('nan'))
        m_val = minimal_agree[a].get(b, float('nan'))
        agree_rows.append({
            'svc_a': a,
            'svc_b': b,
            'judge_agree': j_val,
            'minimal_agree': m_val,
            'delta': j_val - m_val if (pd.notna(j_val) and pd.notna(m_val)) else float('nan'),
        })
agree_long = pd.DataFrame(agree_rows)

print('Judge variant — mean off-diagonal agreement:',
      round(agree_long[agree_long['svc_a'] != agree_long['svc_b']]['judge_agree'].mean(), 3))
print('Minimal variant — mean off-diagonal agreement:',
      round(agree_long[agree_long['svc_a'] != agree_long['svc_b']]['minimal_agree'].mean(), 3))

judge_variant_compare_rows = []
for variant in [v for v in VARIANTS if v != 'judge' and v in variant_agree]:
    rows = []
    for a in _svcs:
        for b in _svcs:
            if a == b:
                continue
            j_val = judge_agree[a].get(b, float('nan'))
            ref_val = variant_agree[variant][a].get(b, float('nan'))
            if pd.notna(j_val) and pd.notna(ref_val):
                rows.append((j_val, ref_val))
    if rows:
        judge_variant_compare_rows.append({
            'reference_variant': variant,
            'mean_offdiag_reference': round(sum(r for _, r in rows) / len(rows), 3),
            'mean_offdiag_judge': round(sum(j for j, _ in rows) / len(rows), 3),
            'judge_minus_reference': round(sum((j-r) for j, r in rows) / len(rows), 3),
        })
judge_variant_compare = pd.DataFrame(judge_variant_compare_rows)
print('Judge compared with every non-judge prompt (mean off-diagonal pairwise agreement):')
display(judge_variant_compare.sort_values('judge_minus_reference', ascending=False))

influence = (agree_long[agree_long['svc_a'] != agree_long['svc_b']]
    .groupby('svc_b')['judge_agree'].mean()
    .sort_values(ascending=False)
    .rename('mean_agreement_from_others')
    .reset_index())
print('Mean agreement FROM others in judge (higher = more influential):')
print(influence.to_string(index=False))


Judge variant — mean off-diagonal agreement: 0.302
Minimal variant — mean off-diagonal agreement: 0.06
Judge compared with every non-judge prompt (mean off-diagonal pairwise agreement):


,reference_variant,mean_offdiag_reference,mean_offdiag_judge,judge_minus_reference
1,fluent_speaker,0.057,0.302,0.245
0,minimal,0.060,0.302,0.241
2,github_searcher,0.083,0.302,0.219


Mean agreement FROM others in judge (higher = more influential):
   svc_b  mean_agreement_from_others
  OpenAI                    0.368875
DeepSeek                    0.348756
   Gemma                    0.315967
  Claude                    0.313204
  Gemini                    0.292244
 Mistral                    0.275297
    Qwen                    0.266460
   Llama                    0.231868


In [28]:
svc_order = _svcs  # consistent ordering

def _heatmap(df, val_col, title, scheme, domain):
    return alt.Chart(df).mark_rect().encode(
        x=alt.X('svc_a:N', sort=svc_order, title='Service'),
        y=alt.Y('svc_b:N', sort=svc_order, title=None),
        color=alt.Color(f'{val_col}:Q',
                        scale=alt.Scale(scheme=scheme, domain=domain),
                        title=val_col),
        tooltip=['svc_a:N', 'svc_b:N',
                 alt.Tooltip(f'{val_col}:Q', format='.2f')],
    ).properties(width=300, height=280, title=title)

heat_judge   = _heatmap(agree_long, 'judge_agree',   '(a) Judge — agreement matrix',   'redyellowgreen', [0, 1])
heat_minimal = _heatmap(agree_long, 'minimal_agree', '(a) Minimal — agreement matrix', 'redyellowgreen', [0, 1])
heat_delta   = _heatmap(agree_long, 'delta',         '(b) Delta: judge − minimal',      'blueorange',    [-0.3, 0.3])

judge_agreement_chart = (heat_minimal | heat_judge | heat_delta).resolve_scale(color='independent')
display(judge_agreement_chart)
save_chart(judge_agreement_chart, FIGURE_DIR / 'judge_vs_minimal_service_agreement.png')

alt.HConcatChart(...)

In [29]:
if 'analysis_terms' not in dir():
    from scripts.utils import apply_manual_analysis_exclusions, load_manual_exclusions
    analysis_terms, _, _ = load_manual_exclusions(os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation'))

# ── Service agreement by language family (judge variant) ────────────────────
# For each language family, which service pairs agree most often in judge?
# Collapsed to: mean pairwise agreement per family × off-diagonal pairs.

judge_vdf2 = load_variant_df(DATA_DIR, TERM_SLUG, 'judge')
if judge_vdf2 is not None:
    judge_vdf2 = apply_manual_analysis_exclusions(judge_vdf2, analysis_terms, list(LLM_TRANS_COLS.values()))
    judge_vdf2 = judge_vdf2.drop_duplicates('language_code')
    judge_vdf2['language_family'] = judge_vdf2['language_code'].apply(get_language_family)

    fam_agree_rows = []
    for fam, grp in judge_vdf2.groupby('language_family'):
        fam_map = {}
        for _, row in grp.iterrows():
            lc = row['language_code']
            fam_map[lc] = {}
            for svc, col in LLM_TRANS_COLS.items():
                val = str(row.get(col, '') or '').strip()
                if val and val.lower() not in ('nan', 'none'):
                    fam_map[lc][svc] = val.lower()
        fam_agree = pairwise_agreement(fam_map)
        off_diag = [(fam_agree[a][b]) for a in _svcs for b in _svcs
                    if a != b and a in fam_agree and b in fam_agree[a]
                    and fam_agree[a][b] == fam_agree[a][b]]
        if off_diag:
            fam_agree_rows.append({'language_family': fam,
                                   'mean_judge_agreement': sum(off_diag)/len(off_diag),
                                   'n_languages': len(grp)})

    fam_agree_df = pd.DataFrame(fam_agree_rows).sort_values('mean_judge_agreement', ascending=False)
    print("Mean pairwise judge agreement by language family (top 15):")
    print(fam_agree_df.head(15).to_string(index=False))

Mean pairwise judge agreement by language family (top 15):
language_family  mean_judge_agreement  n_languages
          Other              0.301584          881
